# Bronze → Silver: dim_calendar

**Propósito:** Leer `dbo.dim_calendar` del lakehouse Bronze, aplicar transformaciones de calidad y escribir en el lakehouse Silver de forma idempotente.

**Transformaciones aplicadas:**
- Cast de `date` a `DateType` (eliminar componente hora)
- Renombrado de `working_day/festive` → `working_day_label` (la barra `/` es problemática en Spark/SQL)
- Drop de `es_habil` (duplicado en español de `is_working_day`)
- Trim de columnas string
- Columna de auditoría `_silver_load_ts`

**Idempotencia:** `overwrite` + `overwriteSchema=true`. Seguro de re-ejecutar.

In [1]:
%run ./config

StatementMeta(, 968cd18e-89bd-4f18-a979-bc5980b0edfe, 3, Finished, Available, Finished, True)

Config cargada → Bronze: lakehouse_poc | Silver: silver_lakehouse_poc | Gold: Gold


In [2]:
BRONZE_TABLE = f"{DEFAULT_SCHEMA}.dim_calendar"
SILVER_TABLE = f"{DEFAULT_SCHEMA}.dim_calendar"

StatementMeta(, 968cd18e-89bd-4f18-a979-bc5980b0edfe, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from datetime import datetime

# Lectura desde Bronze (lakehouse referenciado por nombre de 3 partes)
df_bronze = spark.read.table(f"`{BRONZE_LAKEHOUSE}`.{BRONZE_TABLE}")

print(f"Filas leídas desde Bronze: {df_bronze.count()}")
df_bronze.printSchema()

StatementMeta(, 968cd18e-89bd-4f18-a979-bc5980b0edfe, 5, Finished, Available, Finished, False)

Filas leídas desde Bronze: 4382
root
 |-- date: date (nullable = true)
 |-- complete_date: string (nullable = true)
 |-- year: long (nullable = true)
 |-- month_number: long (nullable = true)
 |-- month: string (nullable = true)
 |-- month_short: string (nullable = true)
 |-- day: long (nullable = true)
 |-- week_day_number: long (nullable = true)
 |-- day_name: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- order_year_month: long (nullable = true)
 |-- quarter: string (nullable = true)
 |-- year_quarter: string (nullable = true)
 |-- order_year_quarter: long (nullable = true)
 |-- semester: string (nullable = true)
 |-- year_semester: string (nullable = true)
 |-- order_year_semester: long (nullable = true)
 |-- first_week_day: date (nullable = true)
 |-- last_week_day: date (nullable = true)
 |-- first_month_day: date (nullable = true)
 |-- last_month_day: date (nullable = true)
 |-- first_day_quarter: date (nullable = true)
 |-- last_day_quarter: date (nulla

In [4]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────

df_silver = (
    df_bronze
    # 1. Cast date → DateType (quitar componente hora)
    .withColumn("date", F.col("date").cast(DateType()))
    .withColumn("first_week_day",   F.col("first_week_day").cast(DateType()))
    .withColumn("last_week_day",    F.col("last_week_day").cast(DateType()))
    .withColumn("first_month_day",  F.col("first_month_day").cast(DateType()))
    .withColumn("last_month_day",   F.col("last_month_day").cast(DateType()))
    .withColumn("first_day_quarter",F.col("first_day_quarter").cast(DateType()))
    .withColumn("last_day_quarter", F.col("last_day_quarter").cast(DateType()))
    .withColumn("first_day_year",   F.col("first_day_year").cast(DateType()))
    .withColumn("last_day_year",    F.col("last_day_year").cast(DateType()))

    # 2. Renombrar columna con barra (problemática en SQL y algunos motores)
    .withColumnRenamed("`working_day/festive`", "working_day_label")

    # 3. Trim de columnas string descriptivas
    .withColumn("month",      F.trim(F.col("month")))
    .withColumn("month_short",F.trim(F.col("month_short")))
    .withColumn("day_name",   F.trim(F.col("day_name")))
    .withColumn("season",     F.trim(F.col("season")))

    # 4. Drop de columna redundante (es_habil == is_working_day)
    .drop("es_habil")

    # 5. Columna de auditoría
    .withColumn("_silver_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

StatementMeta(, 968cd18e-89bd-4f18-a979-bc5980b0edfe, 6, Finished, Available, Finished, False)

In [5]:
# ─── VALIDACIÓN PREVIA A ESCRITURA ────────────────────────────────────────────
row_count = df_silver.count()
null_dates = df_silver.filter(F.col("date").isNull()).count()

print(f"Filas a escribir en Silver: {row_count}")
print(f"Filas con date nulo: {null_dates}")

assert null_dates == 0, "ERROR: hay fechas nulas en dim_calendar"

df_silver.show(3)

StatementMeta(, 968cd18e-89bd-4f18-a979-bc5980b0edfe, 7, Finished, Available, Finished, False)

Filas a escribir en Silver: 4382
Filas con date nulo: 0
+----------+--------------------+----+------------+-----+-----------+---+---------------+---------+----------+----------------+-------+------------+------------------+----------+---------------+-------------------+--------------+-------------+---------------+--------------+-----------------+----------------+--------------+-------------+------+----------+-----------+---------------+-------------------+--------------+--------------+--------+-----------+----------------+---------------+-----------------+------------------+---------------+------+-----------+--------------------+----------------------+------------------+--------------------+
|      date|       complete_date|year|month_number|month|month_short|day|week_day_number| day_name|year_month|order_year_month|quarter|year_quarter|order_year_quarter|  semester|  year_semester|order_year_semester|first_week_day|last_week_day|first_month_day|last_month_day|first_day_quarter|last_da

In [6]:
# ─── ESCRITURA IDEMPOTENTE EN SILVER ──────────────────────────────────────────
# mode=overwrite + overwriteSchema garantiza idempotencia total (re-ejecutable)
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")
)

print(f"✓ Tabla {SILVER_LAKEHOUSE}.{SILVER_TABLE} escrita correctamente con {row_count} filas.")

StatementMeta(, 968cd18e-89bd-4f18-a979-bc5980b0edfe, 8, Finished, Available, Finished, False)

✓ Tabla silver_lakehouse_poc.dbo.dim_calendar escrita correctamente con 4382 filas.
